# Custom brain modules

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

Larvaworld's brain is deliberately made of small, replaceable pieces. A **behavioral module** is a
Python class that reads an input state and writes an output state; the output of one module is the
input of the next, and the whole brain is that graph evaluated once per timestep.

That design has a practical consequence : you can replace any single piece with your own without
touching anything else. This notebook does it twice - a custom olfactor and a custom thermosensor -
and then runs an experiment with them.

**The four steps, every time**

1. Write a class that inherits from the appropriate Larvaworld module base class.
2. Implement `update()` : read `self.input`, compute something, assign `self.output`.
3. Register the class as a **mode** of that module type.
4. Point a model configuration at that mode, and run.

**What you will be able to do afterwards**

- Say what a module receives, what it must produce, and when it is called.
- Write and register a replacement for any brain module.
- Activate it from a model configuration and confirm that it is the one being used.
- Know the difference between overwriting an existing mode and adding a new one.

**Prerequisites** : [The Python API](../1_getting_started/python_api_basics.ipynb) and
[The configuration registry](../4_models_and_environments/configuration_registry.ipynb).

**Cost** : nothing until you switch the simulation on. The custom modules print on every timestep,
so keep the demo short.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_CUSTOM_MODULE_DEMO` | `False` | the simulation using the custom modules |
| `RUN_GUI_DEMO` | `False` | a pygame window for that simulation |
| `SAVE_MEDIA` | `False` | writing it to a video file |

## Setup

The imports include the two module base classes we are about to subclass, and `BrainModuleDB`,
which is the table that maps a module type to its available implementations.

In [1]:
%matplotlib inline

%load_ext param.ipython
import random

import larvaworld as lw
from larvaworld.lib import reg, sim
from larvaworld.lib.model.modules.module_modes import BrainModuleDB
from larvaworld.lib.model.modules.sensor import Olfactor, Thermosensor

lw.VERBOSE = 1

# Tutorial safety switches
RUN_CUSTOM_MODULE_DEMO = False  # runs the simulation with the custom modules
RUN_GUI_DEMO = False  # opens a pygame window
SAVE_MEDIA = False  # writes a video
MEDIA_DIR = "videos"

EXPERIMENT_ID = "chemorbit"

Welcome to the param IPython extension! (https://param.holoviz.org/)
Available magics: %params


Initializing larvaworld registry


Registry configured!


## Section 1 : What a module is

Every brain module type exists in several **modes** - alternative implementations of the same
function. A model configuration does not name a class, it names a mode; the class is looked up in
`BrainModuleDB.BrainModuleModes`. This is the table you extend.

Reading it is the first thing to do : it tells you what already exists, so that you write a new
module only when none of the existing ones fits.

In [2]:
for module, modes in BrainModuleDB.BrainModuleModes.items():
    print(f"{module:14s} {list(modes.keys())}")

crawler        ['constant', 'gaussian', 'square', 'realistic', 'nengo']
interference   ['default', 'square', 'phasic']
turner         ['neural', 'sinusoidal', 'constant', 'nengo']
intermitter    ['default', 'branch']
feeder         ['default', 'nengo']
olfactor       ['default', 'osn']
toucher        ['default']
windsensor     ['default']
thermosensor   ['default']
memory         ['RL', 'MB']


The contract a module has to satisfy is small :

| what | where |
|---|---|
| the sensory or upstream input | `self.input` |
| the value the module produces | `self.output` |
| the agent it belongs to | `self.brain.agent` |
| the simulation it runs in | `self.brain.agent.model` |
| the method called once per timestep | `update()` |

Anything else - state kept between timesteps, references to other modules, random number
generators - is yours to add, because a module is an ordinary Python object.

## Section 2 : Two custom modules

**A custom olfactor.** It reads the currently sensed concentration of the first odor and outputs it
scaled by a random factor. Deliberately meaningless as a model, but it shows every part of the
contract : reading `self.input`, using a derived quantity provided by the base class
(`first_odor_concentration_change`), and writing `self.output`.

**A custom behavior module.** It ignores its input entirely and outputs a random number. It is
registered as a thermosensor, which makes the point that what a module *is* is decided by where you
register it, not by what it computes.

Both print on every call, so you can see them being driven by the simulation.

In [3]:
class CustomOlfactor(Olfactor):
    """An olfactor whose output is the sensed concentration times a random factor."""

    def __init__(self, **kwargs):
        self.last_osn_activity = None
        print("**** CustomOlfactor created ****")
        super().__init__(**kwargs)

    def update(self):
        agent_id = (
            self.brain.agent.unique_id if self.brain is not None else self.agent_id
        )
        sim_id = self.brain.agent.model.id if self.brain is not None else self.sim_id

        # self.input.values() is an array of all odor types, indexed by odor_id.
        # Here we read the values for the first odor:
        olfactory_input = {
            "odor_id": 0,
            # absolute concentration of the 1st odor
            "concentration_mmol": self.input.values()[0],
            # change in concentration of the 1st odor
            "concentration_change_mmol": self.first_odor_concentration_change,
        }

        # The output is what downstream modules receive as input.
        self.output = olfactory_input["concentration_mmol"] * random.random()
        print(
            f"CustomOlfactor output: {self.output} sim_id: {sim_id} agent_id: {agent_id}"
        )


class CustomBehaviorModule(Thermosensor):
    """A module that ignores its input and outputs a random value."""

    def update(self):
        agent_id = (
            self.brain.agent.unique_id if self.brain is not None else self.agent_id
        )
        sim_id = self.brain.agent.model.id if self.brain is not None else self.sim_id

        self.output = random.random()
        print(
            f"CustomBehaviorModule output: {self.output} "
            f"sim_id: {sim_id} agent_id: {agent_id}"
        )

## Section 3 : Registering them

Registration is an assignment into `BrainModuleDB.BrainModuleModes`, and there are two variants
with different consequences :

- **Overwrite an existing mode**, as done here for `olfactor.osn`. Any model that already uses that
  mode now uses your class instead - convenient for a drop-in replacement, but it affects every
  model in the session.
- **Add a new mode**, as done here for `thermosensor.custom`. Nothing changes until a configuration
  asks for the new mode by name, which is the safer option.

`BrainModuleModes` is a class attribute, so the change is global for the rest of the session and
does not persist to disc.

In [4]:
# Replace the built-in 'osn' olfactor implementation with ours
BrainModuleDB.BrainModuleModes.olfactor.osn = CustomOlfactor

# Add a new thermosensor mode rather than replacing the existing one
BrainModuleDB.BrainModuleModes.thermosensor.custom = CustomBehaviorModule

# The same pattern for a turner would be:
# BrainModuleDB.BrainModuleModes.turner.custom = CustomTurnerModule

print("olfactor modes     :", list(BrainModuleDB.BrainModuleModes.olfactor.keys()))
print("thermosensor modes :", list(BrainModuleDB.BrainModuleModes.thermosensor.keys()))

olfactor modes     : ['default', 'osn']
thermosensor modes : ['default', 'custom']


## Section 4 : Activating them in a model

Registering a mode makes it available; a model has to ask for it. That happens in the model
configuration, where each brain module has a `mode` field.

Note the second case : the `navigator` model has no thermosensor at all, so before a mode can be
selected the module itself has to be added. We borrow the section from `thermo_navigator`, a model
that does have one.

In [5]:
exp_conf = reg.conf.Exp.getID(EXPERIMENT_ID)
group_id = exp_conf.larva_groups.keylist[0]
model_id = exp_conf.larva_groups[group_id].model

m_conf = reg.conf.Model.getID(model_id)

print(f"Experiment {EXPERIMENT_ID!r} -> group {group_id!r} -> model {model_id!r}")
print(f"olfactor mode before     : {m_conf.brain.olfactor.mode!r}")
print(f"thermosensor before      : {m_conf.brain.thermosensor}")

Experiment 'chemorbit' -> group 'navigator' -> model 'navigator'
olfactor mode before     : 'default'
thermosensor before      : None


In [6]:
# Use our olfactor implementation
m_conf.brain.olfactor.mode = "osn"

# The navigator has no thermosensor; borrow the section from a model that has one
if m_conf.brain.thermosensor is None:
    m_conf.brain.thermosensor = reg.conf.Model.getID(
        "thermo_navigator"
    ).brain.thermosensor
m_conf.brain.thermosensor.mode = "custom"

print(f"olfactor mode after      : {m_conf.brain.olfactor.mode!r}")
print(f"thermosensor mode after  : {m_conf.brain.thermosensor.mode!r}")

olfactor mode after      : 'osn'
thermosensor mode after  : 'custom'


## Section 5 : Running it

Nothing about the run itself is special - this is the launcher of
[Your first simulation](../1_getting_started/single_simulation.ipynb). If the custom modules are
wired in correctly, their `print` statements appear on every timestep, for every agent.

Keep the duration short : two agents over 0.5 simulated minutes already produce a lot of output.

In [7]:
run_id = "my-custom-modules-run"

screen_kws = {}
if RUN_GUI_DEMO or SAVE_MEDIA:
    screen_kws = {
        "vis_mode": "video",
        "show_display": RUN_GUI_DEMO,
        "save_video": SAVE_MEDIA,
        "fps": 20,
        "video_file": f"larva-sim-{run_id}",
        "media_dir": MEDIA_DIR,
    }

erun = sim.ExpRun(
    experiment=EXPERIMENT_ID,
    modelIDs=["navigator", "Levy_navigator"],
    screen_kws=screen_kws,
    N=2,
    duration=0.5,  # minutes
)

print("Groups in this run :", erun.p.larva_groups.keylist)

Groups in this run : ['navigator', 'Levy_navigator']


In [8]:
if RUN_CUSTOM_MODULE_DEMO:
    erun.simulate()
    print(f"Run {run_id} completed")
else:
    print(
        "Set RUN_CUSTOM_MODULE_DEMO = True to run the simulation with the custom modules."
    )

Set RUN_CUSTOM_MODULE_DEMO = True to run the simulation with the custom modules.


## Where to go next

- [Remote model interface](remote_model_interface.ipynb) - the same idea, but the module runs in
  another process, so it can be written in another framework entirely.
- [Sensory landscapes](../4_models_and_environments/sensory_landscapes.ipynb) - what the sensor
  modules are reading from.
- Reference : [Brain module architecture](../../agents_environments/brain_module_architecture.md)
  and [Module interaction](../../concepts/module_interaction.md), which shows how state flows
  between modules.